### IMPORT

In [1]:
from pathlib import Path

import torch
from torch import nn
import torch.optim as optim
from torch.utils.data import DataLoader
import rootutils

import sys
rootutils.setup_root(Path(".").resolve(), indicator=".project-root", pythonpath=True)
sys.path.append(str(Path(".").resolve().parent / "src"))

from bev.models.bev_emb import BEVAdapter
from bev.models.bevqa import BEVQA
from bev.data.bevqa_dataset import BEVQADataset
from bev.models.head import OutputHead
from bev.models.mca import MCALayer
from bev.models.text_emb import TextAdapter
from bev.training.train import train_epoch, val_epoch

device = "cuda" if torch.cuda.is_available() else "cpu"

### PATH

In [2]:
import hydra
from hydra import initialize, compose
from omegaconf import OmegaConf

try:
    initialize(config_path="../configs", version_base=None)
except:
    pass

cfg = compose(config_name="config", overrides=["paths=mini"])

FEAT_DIR = Path(cfg.paths.bev_features_dir)
DICT_DIR = Path(cfg.paths.dict_dir)
GLOVE = Path(cfg.paths.glove_path)

### DATASET

In [3]:
train_dataset = BEVQADataset(
    bev_dir=FEAT_DIR / "train_mini",
    json_path=DICT_DIR / "NuScenes_train_questions_mini.json",
    glove=GLOVE
)

In [4]:
val_dataset = BEVQADataset(
    bev_dir=FEAT_DIR / "val_mini",
    json_path=DICT_DIR / "NuScenes_val_questions_mini.json",
    glove=GLOVE
)

In [5]:
feat, quest, ans = train_dataset[0]
print(feat.shape, quest.shape, ans)

torch.Size([128, 200, 200]) torch.Size([35, 300]) tensor(26)


### DATALOADER

In [23]:
train_dataloader = DataLoader(train_dataset, batch_size=4, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=4, shuffle=False)

In [24]:
feat, quest, ans = next(iter(train_dataloader))
print(feat.shape, quest.shape, ans)

torch.Size([4, 128, 200, 200]) torch.Size([4, 35, 300]) tensor([25, 21, 22, 17])


### EMBEDDINGS

In [25]:
bev_ad = BEVAdapter() # BEV MAP -> BEV EMB
text_ad = TextAdapter() # [B,35,512] to match BEV emb [B,400,512] 
feat_output = bev_ad(feat)
text_output = text_ad(quest)
print(f"Feat:{feat_output.shape}") # [B,400,512]
print(f"Text:{text_output.shape}") # [B,35,512]

Feat:torch.Size([4, 400, 512])
Text:torch.Size([4, 35, 512])


### MCAN

In [26]:
mca = MCALayer()
bev_out, text_out = mca(feat_output, text_output)
print(bev_out.shape, text_out.shape) # [4, 400, 512], [4, 35, 512]

torch.Size([4, 400, 512]) torch.Size([4, 35, 512])


### HEAD

In [27]:
head = OutputHead()
out = head(text_out)
print(out.shape) # [4,30]

torch.Size([4, 30])


### MODEL

In [28]:
model = BEVQA()
out = model(feat, quest)
print(out.shape) # [4, 30]

torch.Size([4, 30])


### TRAIN

In [ ]:
model = BEVQA().to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-4)
criterion = nn.CrossEntropyLoss()

num_epochs = 10

for epoch in range(num_epochs):
    tr_loss, tr_acc = train_epoch(model, train_dataloader, optimizer, criterion, device)
    val_loss, val_acc = val_epoch(model, val_dataloader, criterion, device)
    print(f"Epoch {epoch+1:02d}/{num_epochs:02d} | "
              f"Train Loss: {tr_loss:.4f} - Acc: {tr_acc:.4f} | "
              f"Val Loss: {val_loss:.4f} - Acc: {val_acc:.4f}")

Epoch 01/10 | Train Loss: 1.5958 - Acc: 0.4043 | Val Loss: 6.4484 - Acc: 0.0378
Epoch 02/10 | Train Loss: 1.1645 - Acc: 0.5012 | Val Loss: 6.8554 - Acc: 0.0727
Epoch 03/10 | Train Loss: 1.0843 - Acc: 0.5321 | Val Loss: 7.2750 - Acc: 0.0414
Epoch 04/10 | Train Loss: 1.0208 - Acc: 0.5627 | Val Loss: 7.1325 - Acc: 0.0681
Epoch 05/10 | Train Loss: 0.9674 - Acc: 0.5874 | Val Loss: 6.9108 - Acc: 0.0488
Epoch 06/10 | Train Loss: 0.9160 - Acc: 0.6043 | Val Loss: 7.7128 - Acc: 0.0580
Epoch 07/10 | Train Loss: 0.8578 - Acc: 0.6317 | Val Loss: 7.1827 - Acc: 0.0562
Epoch 08/10 | Train Loss: 0.7964 - Acc: 0.6569 | Val Loss: 7.2192 - Acc: 0.0820
